# Usage metering tests

In [ ]:
import boto3
import urllib.parse as urlparse 
import json
import shutil
import time

from datetime import datetime, timezone

In [ ]:
PROFILE = 'mp'
REGION = 'us-east-1'

SESSION = boto3.Session(profile_name=PROFILE, region_name=REGION)

In [ ]:
def list_marketplace_products():
    try:
        # Create marketplace catalog client
        mpc = SESSION.client('marketplace-catalog')
        next_token = None
        all_products = []

        paginator = mpc.get_paginator('list_entities')

        pagination_config = {
            'PaginationConfig': {
                'PageSize': 10,   # Number of items per page
            }
        }

        # Define the parameters for the 'list_entities' operation
        operation_parameters = {
            'Catalog': 'AWSMarketplace',  # Required fixed value
            'EntityType': 'SaaSProduct',   # Type of entity to list (e.g., AmiProduct)
            # Add filters or sorting options if needed:
            # 'FilterList': [{'Name': 'EntityId', 'ValueList': ['example-id']}],
            # 'Sort': {'SortBy': 'LastModifiedDate', 'SortOrder': 'DESCENDING'}
        }

        # Paginate through the results
        response_iterator = paginator.paginate(**operation_parameters, **pagination_config)
        #print(response_iterator)
        #print("befor page")
        for page in response_iterator:
            #print(page)
            # Process each page of results
            #print(json.dumps(page.get('EntitySummaryList'), indent=2))
            for entity in page.get('EntitySummaryList'):
                all_products.append(entity)

        return all_products
    except Exception as error:
        print("Error listing marketplace products:", error)
        return []

def get_product_for_product_code(product_code):
    try:
        products = list_marketplace_products()
        mpc = SESSION.client('marketplace-catalog')

        for product in products:
            #print(product)

            product_data = mpc.describe_entity(
                Catalog='AWSMarketplace',
                EntityId=product.get('EntityId')
            )

            print("EntityId:", product.get('EntityId'))
            print("ProductCode:", product_data.get('DetailsDocument').get('Description').get('ProductCode'))
            
            if product_data.get('DetailsDocument').get('Description').get('ProductCode') == product_code:
                print("FOUND Product for code:", product_code, "productId:", product.get('EntityId'))
                return product_data
            
            time.sleep(0.1)
        
        return {}
    except Exception as error:
        print("Error getting id for product code:", error)
        return {}

In [ ]:
product_code = 'cqj79vcg9mig83heufcyjpfai' # My SaaS Product - Contract with Consumption - Landing Page Test 2

#product_code = pp_product_code
#product_code = 'bcmatrsompluro3g7diajhey9'
mpe = SESSION.client('marketplace-entitlement')
response = mpe.get_entitlements(
    ProductCode=product_code,
        
)
print(json.dumps(response,indent=2, default=str))

## Metering into DDB

Put metering records in a DDB table. Replace `table_name` and the value for `dimension` with your product defintion.

In [ ]:
def create_metering_item(customer_aws_account_id, dimension_name ):
    item = {
        "create_timestamp": {
            "N": f"{int(time.time())}"
        },
        "customerIdentifier": {
            "S": customer_aws_account_id
        },
        "dimension_usage": {
            "L": [
            {
                "M": {
                "dimension": {
                    "S": dimension_name
                },
                "value": {
                    "N": "3"
                }
                }
            }
            ]
        },
        "metering_pending": {
            "S": "true"
        }
    }
    
    return item


In [ ]:
table_name = 'AWSMarketplaceMeteringRecords'
ddb = SESSION.client('dynamodb')

In [ ]:
item = create_metering_item('XXX', 'metered_1_id')
print(json.dumps(item, indent=2, default=str))

response = ddb.put_item(
    TableName=table_name,
    Item=item
)
print(json.dumps(response, indent=2, default=str))